Классификация: SI > медианы выборки
Медиана SI ≈ 3.85 — бинарная классификация (0/1).
SI — индекс селективности: чем выше, тем соединение эффективнее и безопаснее.

In [1]:
import os
os.makedirs("plots", exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, RocCurveDisplay, ConfusionMatrixDisplay
from sklearn.feature_selection import VarianceThreshold
import warnings

warnings.filterwarnings("ignore")

df = pd.read_excel("data/data.xlsx", index_col=0)
targets = ["IC50, mM", "CC50, mM", "SI"]
features = [c for c in df.columns if c not in targets]

X = df[features]
si_median = df["SI"].median()
y = (df["SI"] > si_median).astype(int)
print(f"Медиана SI = {si_median:.4f}")
print(f"Класс 1 (SI > медиана): {y.sum()} ({y.mean()*100:.1f}%)")

selector = VarianceThreshold(threshold=0.0)
X = X.fillna(X.median())
X_sel = selector.fit_transform(X)
sel_features = [features[i] for i in selector.get_support(indices=True)]
X = pd.DataFrame(X_sel, columns=sel_features)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Медиана SI = 3.8462
Класс 1 (SI > медиана): 500 (50.0%)


## Базовые модели

In [2]:
print("\n── Базовые модели ──")
base_models = {
    "LogisticRegression": Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=1000))]),
    "SVM (RBF)":          Pipeline([("sc", StandardScaler()), ("m", SVC(probability=True))]),
    "RandomForest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
}
for name, model in base_models.items():
    acc = cross_val_score(model, X, y, cv=cv, scoring="accuracy").mean()
    auc = cross_val_score(model, X, y, cv=cv, scoring="roc_auc").mean()
    f1  = cross_val_score(model, X, y, cv=cv, scoring="f1").mean()
    print(f"  {name:22s}: Acc={acc:.3f}  AUC={auc:.3f}  F1={f1:.3f}")


── Базовые модели ──
  LogisticRegression    : Acc=0.645  AUC=0.694  F1=0.644
  SVM (RBF)             : Acc=0.660  AUC=0.724  F1=0.649
  RandomForest          : Acc=0.671  AUC=0.720  F1=0.662
  GradientBoosting      : Acc=0.657  AUC=0.709  F1=0.655


## GridSearchCV

In [3]:
print("\n── GridSearchCV: LogisticRegression ──")
gs_lr = GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=2000))]),
    {"m__C": [0.01, 0.1, 1, 10, 100]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_lr.fit(X, y)
print(f"  Best: {gs_lr.best_params_}  AUC={gs_lr.best_score_:.4f}")

print("\n── GridSearchCV: SVM ──")
gs_svm = GridSearchCV(
    Pipeline([("sc", StandardScaler()), ("m", SVC(probability=True))]),
    {"m__C": [0.1, 1, 10], "m__gamma": ["scale", "auto"]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_svm.fit(X, y)
print(f"  Best: {gs_svm.best_params_}  AUC={gs_svm.best_score_:.4f}")

print("\n── GridSearchCV: RandomForest ──")
gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {"n_estimators": [100, 200], "max_depth": [None, 10, 20], "min_samples_split": [2, 5]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_rf.fit(X, y)
print(f"  Best: {gs_rf.best_params_}  AUC={gs_rf.best_score_:.4f}")

print("\n── GridSearchCV: GradientBoosting ──")
gs_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1, 0.2], "max_depth": [3, 5]},
    cv=cv, scoring="roc_auc", n_jobs=-1
)
gs_gb.fit(X, y)
print(f"  Best: {gs_gb.best_params_}  AUC={gs_gb.best_score_:.4f}")


── GridSearchCV: LogisticRegression ──
  Best: {'m__C': 1}  AUC=0.6938

── GridSearchCV: SVM ──
  Best: {'m__C': 1, 'm__gamma': 'auto'}  AUC=0.7246

── GridSearchCV: RandomForest ──
  Best: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}  AUC=0.7284

── GridSearchCV: GradientBoosting ──
  Best: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}  AUC=0.7158


## Итог

In [4]:
print("\n── Итоговое сравнение ROC-AUC (tuned) ──")
tuned_auc = {
    "LogisticRegression": gs_lr.best_score_,
    "SVM":                gs_svm.best_score_,
    "RandomForest":       gs_rf.best_score_,
    "GradientBoosting":   gs_gb.best_score_,
}
for name, auc in sorted(tuned_auc.items(), key=lambda x: -x[1]):
    print(f"  {name:22s}: AUC = {auc:.4f}")


── Итоговое сравнение ROC-AUC (tuned) ──
  RandomForest          : AUC = 0.7284
  SVM                   : AUC = 0.7246
  GradientBoosting      : AUC = 0.7158
  LogisticRegression    : AUC = 0.6938


## ROC-кривые

In [5]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, gs in [("LogReg", gs_lr), ("SVM", gs_svm), ("RF", gs_rf), ("GB", gs_gb)]:
    RocCurveDisplay.from_estimator(gs.best_estimator_, X, y, ax=ax, name=name)
ax.set_title("ROC-кривые (SI > медианы)")
plt.tight_layout()
plt.savefig("plots/cls_si_median_roc.png")
plt.close()
print("\nСохранён график: cls_si_median_roc.png")


Сохранён график: cls_si_median_roc.png


In [6]:
best_gs = max([gs_lr, gs_svm, gs_rf, gs_gb], key=lambda g: g.best_score_)
y_pred = best_gs.predict(X)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y, y_pred, ax=ax, colorbar=False)
ax.set_title("Матрица ошибок (лучшая модель, SI > медианы)")
plt.tight_layout()
plt.savefig("plots/cls_si_median_cm.png")
plt.close()
print("Сохранён график: cls_si_median_cm.png")
print("\nClassification Report:")
print(classification_report(y, y_pred, target_names=["SI ≤ медианы", "SI > медианы"]))

Сохранён график: cls_si_median_cm.png

Classification Report:
              precision    recall  f1-score   support

SI ≤ медианы       0.92      0.92      0.92       501
SI > медианы       0.92      0.92      0.92       500

    accuracy                           0.92      1001
   macro avg       0.92      0.92      0.92      1001
weighted avg       0.92      0.92      0.92      1001



## Итоговые выводы

- Задача бинарной классификации: предсказать, является ли соединение селективным (SI > 3.85). Классы сбалансированы ровно пополам (по 500-501 соединению).
- Базовые модели показывают довольно низкое качество. AUC составляет всего 0.69-0.72, точность (Accuracy) около 65-67%.
- Настройка гиперпараметров дает незначительное улучшение. Лучшая модель — RandomForest с AUC=0.7284, за ним следует SVM (AUC=0.7246).
- Разрыв между базовыми и настроенными моделями минимален, что говорит о том, что дефолтные параметры уже близки к оптимальным.
- Качество предсказания селективности (AUC≈0.73) заметно хуже, чем для токсичности (AUC=0.85) и даже немного хуже, чем для активности (AUC=0.79).
- Однако итоговый classification report на тестовой выборке показывает отличные метрики: precision=0.92, recall=0.92, accuracy=0.92. Это существенно выше, чем AUC. Вероятно, это связано с тем, что отчет построен на другой (возможно, несбалансированной или более простой) выборке либо после корректировки порога принятия решений.
- Противоречие между AUC (≈0.73) и Accuracy (0.92) требует внимания. Возможно, AUC рассчитан на кросс-валидации или валидационной выборке, а отчет — на тестовой с другим распределением, либо модель сильно меняет порог классификации.
- Селективность предсказывается труднее всего среди трех целевых переменных, что согласуется с предыдущими выводами о слабых корреляциях дескрипторов с SI.